# Reserve-Aware Value Stacking

This tutorial walks through the `v0.5.0` `da_plus_fcr` workflow. We run the same battery in Belgium and the Netherlands, inspect reserve-aware dispatch outputs, and compare energy-versus-reserve trade-offs.


## What this notebook covers

1. Load the `da_plus_fcr` example configs
2. Run reserve-aware backtests for Belgium and the Netherlands
3. Inspect stable site dispatch outputs
4. Compare reserve share, oracle gap, and throughput


In [ ]:
from __future__ import annotations

import tempfile
from pathlib import Path

import pandas as pd

from euroflex_bess_lab import load_config, run_walk_forward
from euroflex_bess_lab.analytics.reporting import load_report_summary
from euroflex_bess_lab.comparison import compare_runs

ROOT = Path.cwd()
CONFIG_DIR = ROOT / "examples" / "configs"
ARTIFACT_ROOT = Path(tempfile.mkdtemp(prefix="euroflex-v10-reserve-"))
ARTIFACT_ROOT

In [ ]:
belgium = load_config(CONFIG_DIR / "reserve" / "belgium_da_plus_afrr_base.yaml")
canonical = load_config(CONFIG_DIR / "canonical" / "belgium_full_stack.yaml")

belgium.artifacts.root_dir = ARTIFACT_ROOT / "belgium"
canonical.artifacts.root_dir = ARTIFACT_ROOT / "canonical"
belgium.artifacts.save_plots = False
canonical.artifacts.save_plots = False
belgium.artifacts.save_inputs = True
canonical.artifacts.save_inputs = True
belgium.artifacts.save_forecast_snapshots = True
canonical.artifacts.save_forecast_snapshots = True

belgium.run_name = "notebook-belgium-da-afrr"
canonical.run_name = "notebook-belgium-full-stack"

In [ ]:
be_result = run_walk_forward(belgium)
canonical_result = run_walk_forward(canonical)

be_summary = load_report_summary(be_result.output_dir)
canonical_summary = load_report_summary(canonical_result.output_dir)

summary_frame = pd.DataFrame([be_summary, canonical_summary])[
    [
        "market_id",
        "workflow",
        "benchmark_name",
        "total_pnl_eur",
        "reserve_capacity_revenue_eur",
        "reserve_activation_revenue_eur",
        "reserve_share_of_total_revenue",
        "reserved_capacity_mw_avg",
        "oracle_gap_total_pnl_eur",
    ]
]
summary_frame = summary_frame.sort_values(["market_id", "workflow"]).reset_index(drop=True)
summary_frame

## Inspect the reserve-aware dispatch outputs

The dispatch artifact now carries both energy and reserve state. The most important additions are `fcr_reserved_mw`, the remaining headroom columns, and the realized/forecast reserve price columns.

In [ ]:
dispatch_columns = [
    "timestamp_local",
    "charge_mw",
    "discharge_mw",
    "afrr_up_reserved_mw",
    "afrr_down_reserved_mw",
    "reserve_headroom_up_mw",
    "reserve_headroom_down_mw",
    "day_ahead_actual_price_eur_per_mwh",
    "afrr_capacity_up_price_actual_eur_per_mw_per_h",
    "afrr_capacity_down_price_actual_eur_per_mw_per_h",
    "reason_code",
]
be_result.realized_dispatch[dispatch_columns].head(8)

## Compare runs the same way the CLI does

The comparison layer now works across energy-only and reserve-aware runs. Here we compare the two reserve benchmarks and also emit grouped market/workflow tables.

In [ ]:
comparison_dir = compare_runs(
    [be_result.output_dir, canonical_result.output_dir], ARTIFACT_ROOT / "comparison", group_by="workflow"
)
comparison = pd.read_csv(comparison_dir / "comparison.csv")
grouped_workflow = pd.read_csv(comparison_dir / "grouped_by_workflow.csv")

comparison[
    [
        "market_id",
        "workflow",
        "total_pnl_eur",
        "reserve_capacity_revenue_eur",
        "reserve_activation_revenue_eur",
        "reserve_share_of_total_revenue",
        "delta_vs_first_total_pnl_eur",
    ]
]

In [ ]:
grouped_workflow

## Takeaways

- `da_plus_fcr` is intentionally benchmark-first: it models capacity payments and reserve headroom, not live activation dispatch.
- The summary and comparison artifacts now expose reserve share, reserved MW, and reserve-vs-energy tradeoffs directly.
- The same workflow runs through Belgium and Netherlands adapters, which is the foundation for later ancillary-product expansion.